In [1]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import get_body_barycentric_posvel
import sys
sys.path.append('../')
import zodisurf
from zodisurf.zkernelpy_original import earthsun

AU_KM = (1 * u.au).to_value(u.km)

def earth_helio_eq_km(day1990):
    """Earth relative to Sun, ICRS/J2000 equatorial, km. Assumes day 1.0 = 1990-01-01 00:00."""
    t = Time('1990-01-01T00:00:00', scale='utc') + (float(day1990) - 1.0) * u.day
    ep, _ = get_body_barycentric_posvel('earth', t)
    sp, _ = get_body_barycentric_posvel('sun', t)
    return (ep.xyz - sp.xyz).to_value(u.km)

# ---- Test setup: one epoch (obsxyz is one observer position) -----------
day0 = 10000.0
obs_km = earth_helio_eq_km(day0)
print("Earth XYZ (km):", obs_km, "  |r| (AU):", np.linalg.norm(obs_km) / AU_KM)

rng = np.random.default_rng(0)
n = 50
wave = 2.2
lon = rng.uniform(0, 360, n)
lat = rng.uniform(-80, 80, n)
day = np.full(n, day0)

# ---- 1. Python: obsxyz vs default (earthsun) ---------------------------
_, Re_analytic, *_ = earthsun(day, lon, lat)
print("Re analytic vs ephemeris, max |diff| (AU):",
      np.max(np.abs(np.ravel(Re_analytic) - np.linalg.norm(obs_km) / AU_KM)))

zodi_def = zodisurf.get_zmod(wave, 'skysurf', day, lon, lat, engine='numba')
zodi_obs = zodisurf.get_zmod(wave, 'skysurf', day, lon, lat, engine='numba',
                             obsxyz_eq=obs_km)
print("Python obsxyz vs default: max |diff| =", np.max(np.abs(zodi_obs - zodi_def)),
      "  max rel =", np.max(np.abs(zodi_obs / zodi_def - 1)))

Earth XYZ (km): [-8.23733095e+07 -1.16454611e+08 -5.04832102e+07]   |r| (AU): 1.0114642507766267
Re analytic vs ephemeris, max |diff| (AU): 0.00029625002743660644
Python obsxyz vs default: max |diff| = 0.0783699999999996   max rel = 0.0055832768142620726


In [2]:
# Diagnostic: does earthsun's Earth longitude match the ephemeris, and at which epoch offset?
eps = np.deg2rad(23.439291)  # mean obliquity J2000

def ecl_lon_deg(d):
    x, y, z = earth_helio_eq_km(d)
    return np.rad2deg(np.arctan2(y * np.cos(eps) + z * np.sin(eps), x)) % 360

_, _, earth_lon_an, earth_mean_an, _ = earthsun(np.array([day0]), lon[:1], lat[:1])
print("earthsun Earth_Lon (raw):", np.ravel(earth_lon_an)[0],
      "  Earth_Mean_Lon (raw):", np.ravel(earth_mean_an)[0])
print("earthsun Earth_Lon (deg):", np.rad2deg(np.ravel(earth_lon_an)[0]) % 360)  # if radians

for shift in (-1.0, -0.5, 0.0, 0.5, 1.0):
    print(f"ephemeris ecliptic lon at day0{shift:+}: {ecl_lon_deg(day0 + shift):.4f} deg")

earthsun Earth_Lon (raw): 4.141274751260212   Earth_Mean_Lon (raw): 4.116831065124806
earthsun Earth_Lon (deg): 237.27756505129997
ephemeris ecliptic lon at day0-1.0: 236.0536 deg
ephemeris ecliptic lon at day0-0.5: 236.5353 deg
ephemeris ecliptic lon at day0+0.0: 237.0170 deg
ephemeris ecliptic lon at day0+0.5: 237.4985 deg
ephemeris ecliptic lon at day0+1.0: 237.9800 deg


In [3]:
from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic

# A) Ecliptic longitude in the equinox of date vs. the J2000 equinox
def ecl_lon_of_date(d):
    t = Time('1990-01-01T00:00:00', scale='utc') + (float(d) - 1.0) * u.day
    x, y, z = earth_helio_eq_km(d)
    c = SkyCoord(x=x, y=y, z=z, unit='km', representation_type='cartesian', frame='icrs')
    return c.transform_to(BarycentricMeanEcliptic(equinox=t)).lon.deg % 360

print("earthsun Earth_Lon (deg):", np.rad2deg(earth_lon_an[0]) % 360 if np.ndim(earth_lon_an) else np.rad2deg(earth_lon_an) % 360)
print("ephemeris, equinox of date:", ecl_lon_of_date(day0))
print("ephemeris, J2000 equinox:  ", ecl_lon_deg(day0))

# B) Closure test: build obsxyz from earthsun's own Re and Earth_Lon (ecliptic -> J2000 equatorial)
OBL = np.deg2rad(23.439291)   # use your OBLIQUITY constant from zkernel_fast
_, Re_an, Lon_an, _, _ = earthsun(np.array([day0]), lon[:1], lat[:1])
Re_au = float(np.ravel(Re_an)[0])
L = float(np.ravel(Lon_an)[0])
obs_closure = Re_au * AU_KM * np.array([np.cos(L),
                                        np.sin(L) * np.cos(OBL),
                                        np.sin(L) * np.sin(OBL)])
zodi_closure = zodisurf.get_zmod(wave, 'skysurf', day, lon, lat, engine='numba',
                                 obsxyz_eq=obs_closure)
print("closure vs default: max |diff| =", np.max(np.abs(zodi_closure - zodi_def)),
      "  max rel =", np.max(np.abs(zodi_closure / zodi_def - 1)))

earthsun Earth_Lon (deg): 237.27756505129997
ephemeris, equinox of date: 237.25970649630028
ephemeris, J2000 equinox:   237.01697527748934
closure vs default: max |diff| = 3.0000000000640625e-05   max rel = 2.137275799896088e-06


In [4]:
# Make sure numba and numpy agree
z_numba = zodisurf.get_zmod(wave, 'skysurf', day, lon, lat, engine='numba', obsxyz_eq=obs_closure)
z_numpy = zodisurf.get_zmod(wave, 'skysurf', day, lon, lat, engine='numpy', obsxyz_eq=obs_closure)
z_numba/z_numpy

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

## obsxyz_eq validation

**Test 1: Ephemeris Earth position and earthsun distance (Re)**
- Computed Earth's heliocentric position at `day0` from the astropy built-in ephemeris, in J2000 equatorial km (`earth_helio_eq_km`).
- Compared |r| from the ephemeris with `earthsun`'s analytic Re over the test pixels.
- Result: max |diff| = 2.96e-4 AU. This is the expected level of agreement for the analytic approximation.

**Test 2: obsxyz_eq vs. default path (ephemeris-based observer)**
- Ran `get_zmod` with the default `earthsun` geometry, then with `obsxyz_eq` set to the ephemeris Earth position at the same epoch.
- Result: max rel diff = 0.56% (max abs 0.078 MJy/sr).
- Interpretation: This gap is a frame/convention difference, not an error in the obsxyz code path. `earthsun`'s Earth longitude differs from the ephemeris longitude by about 0.26° (J2000 equinox) or about 0.02° (equinox of date).

**Test 3: Ecliptic longitude diagnostic (frame and epoch convention)**
- Compared `earthsun`'s Earth_Lon with the ephemeris ecliptic longitude in two frames:
  - J2000 equinox: 237.0170°
  - Equinox of date: 237.2597°
  - `earthsun` Earth_Lon: 237.2776°
- Result: `earthsun` is closer to the equinox-of-date longitude (about 0.02° off), not the J2000 longitude. The residual is larger than the 0.01° tolerance I had proposed, so the exact frame convention in `earthsun` is not fully pinned down.

**Test 4: Closure test (obsxyz code path, no ephemeris)**
- Built `obsxyz_eq` from `earthsun`'s own Re and Earth_Lon, rotated from ecliptic to J2000 equatorial with the obliquity (23.439291°).
- Ran `get_zmod` with this observer position and compared it with the default path.
- Result: max abs diff = 3.0e-5 MJy/sr, max rel diff = 2.1e-6. This agrees to the level expected from rounding both outputs to 5 decimals.
- Interpretation: The obsxyz branch of `zkernel_fast` reproduces the default geometry when given consistent inputs. The remaining 0.56% difference in Test 2 comes from the frame of the ephemeris input, not from the code.

**Summary:** The obsxyz code path is correct (Test 4). Ephemeris-based observer positions differ from `earthsun` by about 0.02° in longitude because of a frame/equinox convention mismatch (Test 3), which produces the 0.56% gap in Test 2.

Used Claude 5.5 Haiku to perform tests in this notebook.